<link rel="stylesheet" href="berkeley.css">

<h1 class="cal cal-h1">Lecture 03 – CS 189, Fall 2025</h1>

In this lecture, we will cover the basic machine learning lifecycle using a hands-on approach with scikit-learn.
We will work through each stage of the machine learning lifecycle while also introducing standard machine learning tools and techniques.  The machine learning lifecycle consists of four parts:

<div style="text-align: center;">
<img src="https://eecs189.org/fa25/resources/assets/lectures/lec03/images/ml_lifecycle.png" alt="drawing" width="600"/>
</div>


In [2]:
import numpy as np 
import pandas as pd
import plotly.express as px

<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">The Learning Problem</h2>

Suppose we are launching a new fashion trading website where people can upload pictures of clothing they want to trade. We want to help posters identify the clothing in the images. Suppose we have some training data consisting of clothing pictures with labels describing the type of clothing (e.g., "dress", "shirt", "pants").

**What data do we have?**
* Labeled training examples.

**What do we want to predict?**
* The category label of the clothing in the images.  We may want to predict other things as well.

**How would we evaluate success?**
* We likely want to measure our prediction accuracy. 
* We may eventually want to improve accuracy on certain high-value classes.




<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Looking at the Data</h3>

A key step that is often overlooked in machine learning projects is understanding the data. This includes exploring the dataset, visualizing the data, and gaining insights into its structure and characteristics.

We will be using the Fashion-MNIST dataset, which is a (now) classic dataset with gray scale 28x28 images of articles of clothing.

>[Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms.](https://arxiv.org/abs/1708.07747) Han Xiao, Kashif Rasul, Roland Vollgraf.
> https://github.com/zalandoresearch/fashion-mnist

This is an alternative to the even more classic MNIST digits dataset, which contains images of handwritten digits. 

The following block of code will download the Fashion-MNIST dataset and load it into memory. 

In [30]:
# Fetch the Data
import torchvision
data = torchvision.datasets.FashionMNIST(root='data', train=True, download=True)

# Preprocess the data into numpy arrays
images = data.data.numpy().astype(float)
targets = data.targets.numpy() # integer encoding of class labels
class_dict = {i:class_name for i,class_name in enumerate(data.classes)}
labels = np.array([class_dict[t] for t in targets]) # raw class labels
n = len(images)

print("Loaded FashionMNIST dataset with {} samples.".format(n))
print("Classes: {}".format(class_dict))
print("Image shape: {}".format(images[0].shape))
print("Image dtype: {}".format(images[0].dtype))
print("Image 0:\n", images[0])
print("Labels: {}".format(labels))

Loaded FashionMNIST dataset with 60000 samples.
Classes: {0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat', 5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'}
Image shape: (28, 28)
Image dtype: float64
Image 0:
 [[  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
    0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
    0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
    0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.   0.
    0.  13.  73.   0.   0.   1.   4.   0.   0.   0.   0.   1.   1.   0.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   3.   0.
   36. 136. 127.  62.  54.   0.   0.   0.   1.   3.   4.   0.   0.   3.]
 [  0.   0.   0.   0.   0.  

<link rel="stylesheet" href="berkeley.css">


<h4 class="cal cal-h4">Understanding The Raw Features (Images)</h4>

How much data do we have?

In [4]:
images.shape

(60000, 28, 28)

In [5]:
images.size

47040000

The images are stored in a 60000 by 28 by 28 tensor.  This means we have 60000 images, each of which is 28 pixels wide and 28 pixels tall.  Each pixel is represented by a single value.  What are those values?


In [22]:
counts, bins =  np.histogram(images, bins=np.arange(257))
print(bins)
fig_pixels = px.bar(x=bins[:-1], y=counts, title="Pixel value distribution", 
       log_y=True, labels={"x":"Pixel value", "y":"Count"})

print(counts)
fig_pixels

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

In [26]:
counts_cal = [0] * 256

for image in images:
    for r in image:
        for p in r:
            counts_cal[int(p)] += 1

assert(counts_cal == counts.tolist())

In [21]:
counts, bins =  np.histogram(images, bins=255)
fig_pixels = px.bar(x=bins[1:], y=counts, title="Pixel value distribution", 
       log_y=True, labels={"x":"Pixel value", "y":"Count"})
fig_pixels

print(counts)

[23616498   477616   286907   191709   131597    96949    75610    61138
    52858    48514    46274    44886    44885    46283    45735    46711
    45039    47566    46710    46971    47733    47792    48326    48138
    48987    49532    49848    50022    49935    50072    51863    51011
    52778    51508    53662    53381    54386    53205    54569    54075
    55591    55461    54597    55734    54812    57583    56250    56629
    58381    56393    52756    62900    58471    58171    57973    60289
    58523    60188    59338    59099    61320    60600    59640    58578
    61588    61618    61618    58991    63870    60733    63088    61540
    62224    63560    60593    65295    62925    63640    64095    63593
    64652    64854    64301    64933    51808    76308    65725    66889
    65157    64161    68260    65593    66104    67969    66702    67876
    68340    68175    67043    67354    69163    60867    75601    69324
    67847    70404    69597    69781    71638    69

It is important to learn how to visualize and work with data.  Here we use Plotly express to visualize the image. Note, I am using the 'gray_r' color map to visualize the images, which is a gray scale color map that is reversed (so that black is 1 and white is 0). 

In [28]:
px.imshow(images[0], color_continuous_scale='gray_r') 

The following snippet of code visualizes multiple images in a grid.  You are not required to understand this code, but it is useful to know how to visualize images in Python. 


In [35]:
def show_images(images, max_images=40, ncols=5, labels = None):
    """Visualize a subset of images from the dataset.
    Args:
        images (np.ndarray): Array of images to visualize [img,row,col].
        max_images (int): Maximum number of images to display.
        ncols (int): Number of columns in the grid.
        labels (np.ndarray, optional): Labels for the images, used for facet titles.
    Returns:
        plotly.graph_objects.Figure: A Plotly figure object containing the images.
    """
    n = min(images.shape[0], max_images) # number of images to show
    px_height = 220 # height of each image in pixels
    fig = px.imshow(images[:n, :, :], color_continuous_scale='gray_r', 
                    facet_col = 0, facet_col_wrap=ncols,
                    height = px_height * int(np.ceil(n/ncols)))
    fig.update_layout(coloraxis_showscale=False)
    if labels is not None:
        # Extract the facet number and replace with the label.
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))

    return fig

In [36]:
show_images(images, 20, labels=labels)

Let's look at a few examples of each class. Here we use Pandas to group images by their labels and sample 2 for each class. You are not required to know Pandas (we won't test you on it), but it is a useful library for data manipulation and analysis and we will use it often in this course.

In [43]:
idx = (
    pd.DataFrame({"labels": labels})
      .groupby("labels", as_index=False)
      .sample(2)
      .index
      .to_numpy())
show_images(images[idx,:,:], labels=labels[idx])

<link rel="stylesheet" href="berkeley.css">


<h4 class="cal cal-h4">Understanding the Labels</h4>

New let's examine the labels.  Are they discrete?  What is the distribution? Are there missing values or errors?

In the Fashion-MNIST dataset, each image is labeled with a class corresponding to a type of clothing. There are 10 classes in total. 

However, it is also important to understand the distribution of labels in the dataset. This can help us identify potential issues such as class imbalance, where some classes have significantly more samples than others.

In [44]:
labels

array(['Ankle boot', 'T-shirt/top', 'T-shirt/top', ..., 'Dress',
       'T-shirt/top', 'Sandal'], shape=(60000,), dtype='<U11')

The labels are strings (discrete).

What is the distribution of labels?

In [45]:
px.histogram(labels, title="Label distribution")   

There appear to be equal proportion of each type of clothing.  We don't have any missing values since all labels are one of the 10 classes (no blank or "missing" label values).

Most real world datasets aren't this balanced or clean.  In fact, it's common to see a long tail distribution, where a few classes are very common and many classes are rare.  

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Reviewing the Learning Setting</h3>

Having examined the data we can see that we have a large collection of pairs of features and categorical labels (with 10 classes). This is a **supervised learning** problem, where the goal is to learn a mapping from the input features (images) to the output labels (categories). Because the labels are discrete this is a **classification** problem. 

It is also worth noting that because the input features are images this is also a **computer vision** problem.  This means that when we get to the model development stage, we will need to consider techniques that are specifically designed for multi-class classification and in particular computer vision.

<br><br><br>

---

**Return to Slides**

---

<br><br><br>

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Train-Test-Validation Split</h3>

We will split the dataset into a training set, a validation set, and a test set. The training set will be used to train the model, while the validation set will be used to tune the model's hyperparameters. The test set will be used to evaluate the model's performance. 

Technically, the Fashion-MNIST dataset has a separate test set, but we will demonstrate how to split data in general.

In [46]:
# use sklearn to construct a train test split
from sklearn.model_selection import train_test_split

In [47]:
# Construct the train - test split
images_tr, images_te, labels_tr, labels_te = train_test_split(
    images, labels, test_size=0.2, random_state=42)

# Construct the train - validation split
images_tr, images_val, labels_tr, labels_val = train_test_split(
    images_tr, labels_tr, test_size=0.2, random_state=42)

print("images_tr shape:", images_tr.shape)
print("images_val shape:", images_val.shape)
print("images_te shape:", images_te.shape)

images_tr shape: (38400, 28, 28)
images_val shape: (9600, 28, 28)
images_te shape: (12000, 28, 28)


<br><br><br>

---

**Return to Slides**

---

<br><br><br>

<link rel="stylesheet" href="berkeley.css">


<h2 class="cal cal-h2"> Model Design </h2>

We have already loaded the Fashion-MNIST dataset in the previous section. Now, we will preprocess the data to make it suitable for training a classification model. 




<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Feature Engineering</h3>

**Feature Engineering** is the process of transforming the raw features into a representation that can be used effectively by machine learning techniques.  This will often involve transforming data into vector representations. 

<h4 class="cal cal-h4">Naive Image Featurization</h4>

For this example we are working with low-resolution grayscale image data. Here we adopt a very simple featurization approach -- flatten the image. We will convert the 28x28 pixel images into 784-dimensional vectors.

In [48]:
images_tr.shape

(38400, 28, 28)

In [49]:
def flatten(images):
    return images.reshape(images.shape[0], -1)

In [50]:
X_tr = flatten(images_tr)

In [51]:
X_tr.shape

(38400, 784)

<link rel="stylesheet" href="berkeley.css">

<h4 class="cal cal-h4">Standardization</h4>

Recall that the pixel intensities are from 0 to 255:


In [52]:
fig_pixels

Let's standardize the pixel intensities to have zero mean and unit variance.

Here we use the sklearn StandardScaler

In [53]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize a StandardScaler object
image_scaler = StandardScaler()

# 2. Fit the scaler
image_scaler.fit(flatten(images_tr))

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


What do the mean and variance images tell us about the dataset?

In [54]:
display(px.imshow(image_scaler.mean_.reshape(28,28), 
                  color_continuous_scale='gray_r', title="Mean image"))
display(px.imshow(image_scaler.var_.reshape(28,28), 
                  color_continuous_scale='gray_r', title="Variance image"))

Let's create a generic featurization function that we can reuse for different datasets.  Notice that this function uses the image_scaler that we fit to the training data.


In [55]:
def featurizer(images):
    flattened = flatten(images)
    return image_scaler.transform(flattened)

X_tr = featurizer(images_tr)

Our new images look similar to the original images but they have been standardized to have zero mean and unit variance. This should help improve the performance of our machine learning models.

In [56]:
show_images(X_tr.reshape(images_tr.shape), max_images=10, labels=labels_tr)

<link rel="stylesheet" href="berkeley.css">

<h4 class="cal cal-h4">One-Hot Encoding</h4>

We don't need to one-hot encode the features in this dataset so we will briefly demonstrate on another dataset:

In [57]:
df = pd.DataFrame({"color": ["red", "green", "red", "blue", "blue", "yellow", ""]})
df

,color
0,red
1,green
2,red
3,blue
4,blue
5,yellow
6,


In [58]:
from sklearn.preprocessing import OneHotEncoder
# 1. Initialize a OneHotEncoder object
ohe = OneHotEncoder()
# 2. Fit the encoder
ohe.fit(df[["color"]])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [59]:
ohe.categories_

[array(['', 'blue', 'green', 'red', 'yellow'], dtype=object)]

In [60]:
ohe.transform(df[["color"]]).toarray()
ohe.categories_

[array(['', 'blue', 'green', 'red', 'yellow'], dtype=object)]

<link rel="stylesheet" href="berkeley.css">

<h4 class="cal cal-h4">Bag of Words</h4>

We also don't need the bag-of-words representation for this dataset, but we will demonstrate it briefly using another dataset.

In [61]:
df['text'] = [
    "Red is a color.",
    "Green is for green food.",
    "Red reminds me of red food.",
    "Blue is my favorite color!",
    "Blue is for Cal!",
    "Yellow is also for Cal!",
    "I forgot to write something."
]

In [62]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize a CountVectorizer object
vectorizer = CountVectorizer()

# 2. Fit the vectorizer
vectorizer.fit(df["text"])


,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [63]:
pd.DataFrame(vectorizer.transform(df["text"]).toarray(), 
             columns=vectorizer.get_feature_names_out())

,also,blue,cal,color,favorite,food,for,forgot,green,is,me,my,of,red,reminds,something,to,write,yellow
0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,1,1,0,2,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,1,0,1,2,1,0,0,0,0
3,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0
4,0,1,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0
5,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1
6,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0


<br><br><br>

---

**Return to Slides**

---

<br><br><br>



<link rel="stylesheet" href="berkeley.css">


<h2 class="cal cal-h2">Modeling and Optimization</h2>

In this section, we will go through the modeling process. We will focus on developing a classification model.

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Training (Fitting) a Classifier </h3>

We will start with the most basic classifier, the logistic regression model, to demonstrate the classification workflow.

Logistic regression is a **linear model** that is commonly used for binary and multi-class classification tasks.  It is also a good starting point for understanding more complex deep learning models that will be covered later in the course.

Here we use `sklearn` to fit a logistic regression model to the training data. The `LogisticRegression` class from `sklearn.linear_model` is used to create an instance of the model. 

The `fit` method is called on the model instance, passing in the training data and labels. This trains the model to learn the relationship between the input features (flattened images) and the target labels (clothing categories).  In scikit-learn, the `fit` method is used to train any model on the provided data.


In [64]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()
lr_model.fit(X=X_tr, y=labels_tr)

/Users/kyzyc/Projects/CS189_2025Fall_GA_Template/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Notice that we get a warning that: 
```plaintext
lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
```

This warning indicates that the **optimization algorithm** used by the logistic regression model did not converge to a solution within the default number of iterations. This can happen if the model is complex or if the data is not well-scaled.

We can change the optimization algorithm used by the logistic regression model. The default algorithm is `lbfgs`, which is a quasi-Newton method. Other options include `newton-cg`, `sag`, and `saga`. Each of these algorithms has its own strengths and weaknesses, and the choice of algorithm can affect the convergence speed and final performance of the model.


In this class, we will explore variations of stochastic gradient descent like `saga`. Let's try using the `saga` algorithm instead of `lbfgs` to see if it converges faster.  Here we will also set the `tol` parameter since we don't want to wait.



In [65]:
lr_model = LogisticRegression(tol=0.05, solver='saga', random_state=42)
lr_model.fit(X=X_tr, y=labels_tr)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.05
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicl

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Parameters </h3>

The **parameters** of a model are the internal variables that the model learns during the training process. For example, in logistic regression, the parameters are the weights assigned to each feature. These weights are adjusted during training to minimize the loss function, which measures how well the model's predictions match the actual labels.

In [66]:
print("model.coef_.shape:", lr_model.coef_.shape)
print("model.intercept_.shape:", lr_model.intercept_.shape)
print(lr_model.coef_)
print(lr_model.intercept_)

model.coef_.shape: (10, 784)
model.intercept_.shape: (10,)
[[ 9.90445492e-05 -6.36010321e-03 -3.85913757e-03 ...  1.49595978e-03
   4.32587736e-03  2.73064427e-03]
 [-1.89699677e-03  6.63981322e-03 -7.47402405e-03 ... -1.80362390e-02
  -1.84524048e-02 -1.52846533e-02]
 [-7.43264082e-04 -4.46484428e-03  1.60046019e-03 ...  5.50850058e-03
   1.38217724e-02  3.68772263e-03]
 ...
 [ 1.77951137e-04 -1.08447436e-03 -8.59128688e-04 ... -5.03797045e-03
  -1.78474348e-03 -2.49799352e-03]
 [-8.24675871e-03 -2.41366751e-03 -7.31344979e-03 ... -1.80925346e-02
  -1.21750176e-02 -1.40790500e-03]
 [-1.69500201e-04  7.44355526e-05 -5.08764217e-03 ... -5.67020645e-03
  -1.32036337e-03 -1.92847951e-03]]
[-0.03276315  0.01498156  0.02486191 -0.00368395  0.04477243 -0.06132416
  0.09267479 -0.03592529 -0.00330317 -0.04029096]


We can also visualize these coefficients.  This can help us understand which pixels are most important for each class. We will learn more about this model in the future. (You don't have to understand these details now.)

In [67]:
coeffs = lr_model.coef_
show_images(coeffs.reshape(10, 28, 28), labels=lr_model.classes_)

#### Neural Networks


Neural networks are often the model of choice for image classification tasks. They can learn complex patterns in the data and often outperform simpler models like logistic regression. However, they also require the correct architecture and significant training data and computational resources. 

Here we will try a simple neural network with two hidden layers.

In [68]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50), 
    max_iter=100, tol=1e-3, random_state=42)

mlp.fit(X=X_tr, y=labels_tr)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Hyperparameters </h3>

The **hyperparameters** are the arguments that are set before the training process begins. These include the choice of optimization algorithm, the learning rate, and the number of iterations, among others. Hyperparameters are typically tuned using techniques like cross-validation to find the best combination for a given dataset.  

Confusingly these hyperparameters are often referred to as "parameters" in the context of machine learning libraries like `sklearn`. For example, the `LogisticRegression` class has hyperparameters like `solver`, `C`, and `max_iter` that can be adjusted to improve model performance.

In [69]:
lr_model

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.05
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicl

To evaluate the model we will use the validation dataset.

In [70]:
X_val = featurizer(images_val)

Let's try tuning the regularization parameter `C`.  To make this process more illustrative we will work with a smaller subset of the training data (n=1000).  This will allow us to better demonstrate underfitting and overfitting and significantly speed up the training process.


In [71]:
n_small = 1000
X_tr_small = X_tr[:n_small,:]
labels_tr_small = labels_tr[:n_small]

In practice, we need to be careful when tuning regularization parameters against a sample of the data.  In this case, more regularization is likely needed for smaller datasets to prevent overfitting.

In [ ]:
from sklearn.metrics import log_loss

C_vals = np.logspace(-5, 1, 20)

print(C_vals)

logprob_tr = []
logprob_val = []
acc_tr = []
acc_val = []

for C in C_vals:
    print("starting training with C =", C)
    model = LogisticRegression(tol=1e-3, random_state=42, C=C)
    model.fit(X=X_tr_small, y=labels_tr_small)
    
    # compute the logprob accuracy
    logprob_tr.append(-log_loss(labels_tr_small, model.predict_proba(X_tr_small), labels=model.classes_))
    logprob_val.append(-log_loss(labels_val, model.predict_proba(X_val), labels=model.classes_))

    # compute the accuracy
    acc_tr.append(np.mean(model.predict(X_tr_small) == labels_tr_small))
    acc_val.append(np.mean(model.predict(X_val) == labels_val))


[1.00000000e-05 2.06913808e-05 4.28133240e-05 8.85866790e-05
 1.83298071e-04 3.79269019e-04 7.84759970e-04 1.62377674e-03
 3.35981829e-03 6.95192796e-03 1.43844989e-02 2.97635144e-02
 6.15848211e-02 1.27427499e-01 2.63665090e-01 5.45559478e-01
 1.12883789e+00 2.33572147e+00 4.83293024e+00 1.00000000e+01]
starting training with C = 1e-05
starting training with C = 2.06913808111479e-05
starting training with C = 4.281332398719396e-05
starting training with C = 8.858667904100833e-05
starting training with C = 0.00018329807108324357
starting training with C = 0.000379269019073225
starting training with C = 0.0007847599703514606
starting training with C = 0.001623776739188721
starting training with C = 0.003359818286283781
starting training with C = 0.0069519279617756054
starting training with C = 0.01438449888287663
starting training with C = 0.029763514416313162
starting training with C = 0.06158482110660261
starting training with C = 0.1274274985703132
starting training with C = 0.263665

In [73]:
df_logprob = pd.DataFrame({
    "C_val": C_vals, 
    "Train": logprob_tr, "Validation": logprob_val,
}).set_index("C_val") 

display(
    px.line(df_logprob, 
        labels={"value": "Avg. Log Prob.", "C_val": "Reg. Parameter C"},
        title="LR Classifier Log Prog vs Reg. Parameter",
        markers=True,
        log_x=True,
        width=800, height=500)
)

df_acc = pd.DataFrame({
    "C_val": C_vals, 
    "Train": acc_tr, "Validation": acc_val
}).set_index("C_val") 

display(
    px.line(df_acc, 
        labels={"value": "Accuracy", "C_val": "Reg. Parameter C"},
        title="LR Classifier Accuracy vs Reg. Parameter",
        markers=True,
        log_x=True,
        width=800, height=500
    )
)

<link rel="stylesheet" href="berkeley.css">


<h2 class="cal cal-h2">Evaluating the Model</h2>

After training the model, we can use it to make predictions on new data. The `predict` method of the trained model is used to generate predictions based on the input features.

Let's return to our logistic regression model.

In [74]:
lr_model.predict(X_tr[:10,:])

array(['Ankle boot', 'Coat', 'Ankle boot', 'T-shirt/top', 'Pullover',
       'Ankle boot', 'Dress', 'T-shirt/top', 'Coat', 'Sandal'],
      dtype='<U11')

Do you agree with the predictions? Let's visualize the predictions on a few test images.

In [78]:
show_images(images_tr[:10,:].reshape(10, 28, 28),
            labels = lr_model.predict(X_tr[:10,:]))


In [79]:
show_images(images_tr[:10,:].reshape(10, 28, 28),
            labels = labels_tr[:10])

Now let's see what the correct labels are for these images. 

In [80]:
k = 10
tmp_labels = labels_tr[:k] + " (pred=" + lr_model.predict(X_tr[:k,:]) + ")"
show_images(images_tr[:k,:].reshape(k, 28, 28), labels=tmp_labels)

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Predicting Probabilities</h3>

Many models can also provide probabilities for each class using the `predict_proba` method. This is useful for understanding the model's confidence in its predictions.  In this class, we will often use a probabilistic framing, where we interpret the output of the model as probabilities of each class.

In [81]:
lr_model.predict_proba(X_tr[:5,:])

array([[9.51642010e-01, 1.25766964e-03, 8.94327054e-07, 3.11486837e-07,
        2.59469407e-07, 7.35306052e-03, 1.78581053e-07, 3.97435123e-02,
        5.22696365e-07, 1.58056694e-06],
       [1.05752334e-04, 2.47086595e-03, 8.32072100e-01, 3.44094506e-03,
        4.58904789e-02, 4.24209010e-05, 1.13951633e-01, 1.94167275e-05,
        1.55456636e-03, 4.51820449e-04],
       [9.30078717e-01, 1.40564685e-03, 1.91994009e-06, 6.84403412e-07,
        8.73522603e-07, 9.24863395e-03, 8.71969951e-07, 5.92557003e-02,
        1.75317539e-06, 5.19911487e-06],
       [2.05798117e-04, 2.21031435e-05, 6.61559787e-04, 2.37402967e-02,
        2.18046824e-02, 2.07214721e-05, 3.03973688e-02, 1.53740432e-04,
        9.18742226e-01, 4.25150290e-03],
       [2.23839377e-04, 9.60149725e-02, 1.08016253e-01, 2.85455925e-04,
        7.19252331e-01, 1.85483941e-03, 7.24570316e-02, 3.23939232e-04,
        1.53916187e-03, 3.21759981e-05]])

We can visualize these probabilities for the same images we predicted earlier.

In [83]:
k = 10
df = pd.DataFrame(lr_model.predict_proba(X_tr[:k,:]), columns=lr_model.classes_)
bars = px.bar(df, barmode='stack',orientation='v')
bars.update_layout(xaxis_tickmode='array', xaxis_tickvals=np.arange(k))
display(bars)

<link rel="stylesheet" href="berkeley.css">


<h3 class="cal cal-h3">Accuracy Metrics and Test Performance</h3>

After training the model, we often want to evaluate the model. There are many ways to evaluate a model, and the best method depends on the task and the data. For classification tasks, we often use metrics like accuracy, precision, recall, and F1-score. Let's start with accuracy. 

Accuracy is the simplest metric, which measures the proportion of correct predictions out of the total number of predictions. 

Let's start by computing the accuracy of our model on the training set.

In [84]:
np.mean(lr_model.predict(X_tr) == labels_tr, axis=0)

np.float64(0.8488541666666667)

One of the issues with the training set is that the model may have overfit to the training data, meaning it performs well on the training set but poorly on unseen data. Intuitively, this is like practicing on a set of questions and then getting those same questions right on a test, but not being able to answer new questions that are similar but not identical.

To assess the model's performance on unseen data, we will evaluate it on the test set. Recall, the test set is a separate portion of the dataset that was not used during training.

In [85]:
X_te = featurizer(images_te)

In [86]:
np.mean(lr_model.predict(X_te) == labels_te, axis=0)

np.float64(0.84075)

In [87]:
from sklearn.metrics import accuracy_score

train_acc = accuracy_score(labels_tr, lr_model.predict(X_tr))
val_acc = accuracy_score(labels_val, lr_model.predict(X_val))
test_acc = accuracy_score(labels_te, lr_model.predict(X_te))

print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)
print("Test accuracy:", test_acc)

Train accuracy: 0.8488541666666667
Validation accuracy: 0.849375
Test accuracy: 0.84075


The test accuracy is slightly lower than the training accuracy, which is expected. However, the difference is not too large, indicating that the model has not overfit significantly.

**Is this accuracy good? What would a random guess yield?**

A common way to evaluate a classification model is to compare its accuracy against a baseline. Perhaps the simplest baseline is random guessing, where we randomly assign a class to each image.

**What accuracy does random guessing yield?**

This would depend on the how frequently each class appears in the test dataset. 

In [88]:
np.random.seed(42)
print("Model Accuracy:", np.mean(lr_model.predict(X_val) == labels_val, axis=0))
print("Random Guess Accuracy:", 
      np.mean(np.random.choice(lr_model.classes_, size=len(labels_te)) == labels_te, axis=0))

Model Accuracy: 0.849375
Random Guess Accuracy: 0.10108333333333333


Does our model struggle with any particular class?

In [89]:
isWrong = lr_model.predict(X_val) != labels_val
# make a histogram with frequency of correct and incorrect predictions
fig = px.histogram(labels_val[isWrong], histnorm='percent')
fig.update_layout(xaxis_title="Label", 
                  yaxis_title="Percentage of Incorrect Predictions")
fig.update_xaxes(categoryorder="total descending")

For classification tasks, we often want to look at more than just accuracy. We can use a confusion matrix to visualize the performance of the model across different classes. The confusion matrix shows the number of correct and incorrect predictions for each class.

In [90]:
from sklearn.metrics import confusion_matrix

fig = px.imshow(
    confusion_matrix(labels_val, lr_model.predict(X_val)), 
    color_continuous_scale='Blues'
    )
fig.update_layout(
        xaxis_title="Predicted Label",
        yaxis_title="True Label",
        coloraxis_showscale=False,
        xaxis=dict(tickmode='array', tickvals=np.arange(len(model.classes_)), ticktext=model.classes_),
        yaxis=dict(tickmode='array', tickvals=np.arange(len(model.classes_)), ticktext=model.classes_)
    )   


<link rel="stylesheet" href="berkeley.css">


<h2 class="cal cal-h2">Last Thoughts</h2>

In the homework, you will have a chance to work with this data and use scikit-learn in more depth.  We recommend reading the documentation and tutorial on the scikit-learn website as we go through the course.  These provide a great resource for understanding the various functions and capabilities of the library as well as machine learning concepts.
